# Computer Graphics

**Computer graphics** is the field dedicated to generating, manipulating, and synthesising visual content with computers.

The primary goal is to transform an abstract description of a scene (shapes, materials, lights, camera) into a compelling 2D image.

## The rendering pipeline

The rendering pipeline describes how a 3D scene is converted into a 2D image on the screen. These stages reflect the hardware-level process used by the GPU.

The stages are:

1. **Object transformation** - converts object coordinates from local space to world space
3. **Clipping** - removes geometry outside the view
4. **Backface culling** - discards polygons facing away from the camera
5. **Rasterisation** - determines which screen pixels are covered by triangles, producing fragments (potential pixels)
6. **Hidden surface removal (HSR) & Shading** - depth testing to keep only the closest fragment per pixel and computation of final colour

### Object transformation

Object transformation is the process of taking a model defined in its own coordinate system (local space) and placing it correctly into the scene (world space).

In the local space, each object is defined relative to its own centre (origin). 

- This essentially provides a template or blueprint for the object, which can then be reused easily


In the global space, all objects are placed into a shared global coordinate system, so positions are relative to the world origin.

- Multiple copies of the same object can be placed in different locations, and objects can interact with other objects in the scene

The relation between the local and global space is given by:
$$
P_{world} = M \times P_{local}
$$
where:

- $P_{local}$ is the vertex in object (local) space
- $M$ is the transformation matrix (model matrix)
- $P_{world}$ is the vertex in world space

$M$ is a $4 \times 4$ matrix which combines translation, rotation and scaling. The position of each vertex is calculated independently, using SIMD (Single Instruction, Multiple Data) on the GPU  to transform thousands of vertices in parallel (at the same time).

Vertices are represented in homogeneous coordinates as $(x,y,x,w)$ where $w=1$ for positions, allowing translation to be included in matrix multiplication, while direction vectors use $w=0$ so they are unaffected by translation. After applying the projection matrix, 
$w$ encodes depth-related information of the vertex.

### Perspective projection

Perspective projection is the process of simulating a camera lens by transforming 3D coordinates into a 2D representation, where distant objects appear smaller.

To render a 3D scene:

- a **virtual camera** is defined, which establishes the point from which the scene is being viewed
- this camera defines a specific volume of space called the **view frustum**
  - the view frustum is a truncated pyramid that determines exactly what is visible
  - only objects that fall inside the view frustum are rendered
  - the view frustum is limited by:
    - Field of View (FOV)
    - the Near Plane (closest visible distance)
    - the Far  Plane (draw distance)
- a **projection matrix** transforms coordinates into **clip space**, mapping the view frustum into a unit cube
- a **perspective divide** is applied after clipping, where the $x$, $y$, and $z$ coordinates are divided by the homogeneous coordinate $w$, which encodes depth information generated by the projection matrix
  - this causes objects further away to appear smaller, creating the illusion of depth

![Perspective Projection](perspective_projection.png)

### Clipping

Clipping is the process of removing geometry that lies outside the camera's view (view frustum) to reduce the amount of processing needed. 

The GPU checks whether vertices/primitives lie within the viewing volume, keeping only those that are within the view: 

- if a triangle is partly inside and partly outside the view frustum, it cannot be discarded (as part is visible) but it cannot be rendered in full (as part is outside the view)
- the solution is to cut the triangle along the clipping boundary and create new vertices where edges intersect the frustum, which forms new triangles which fit entirely within the view volume
  - a common algorithm that does this is Sutherland-Hodgman

![Clipping](clipping.png)

Clipping is performed after the projection matrix is applied and before the perspective divide, as perspective projection transforms the view frustum into a unit cube in clip space. This makes it easier to check if points lie within the cube; simply check that:

- $-w \le x \le w$
- $-w \le y \le w$
- $-w \le z \le w$

### Backface culling

Backface culling is the process of discarding polygons that face away from the camera, since they are not visible in a closed 3D object, so rendering them is computationally wasteful. 

To determine if a surface is facing the camera, a dot product can be used:
$$ \vec{V} \cdot \vec{N}$$
where:
- $\vec{V}$ is the view direction
- $\vec{N}$ is the surface normal vector

If the dot product is positive, the face is pointing away, so the surface is discarded. If the dot product is negative, the face is pointing towards the camera, and is kept. 

A slightly more efficient way of determining whether triangles are back-facing is by checking **winding order** after projection:

- winding order determines whether a triangle is front- or back-facing by checking whether its vertices appear clockwise or counter-clockwise in screen space
- one winding direction is defined as front-facing and the opposite is treated as back-facing, allowing fast culling without computing normals

### Rasterisation

Rasterisation is the process of converting continuous 2D geometric shapes (triangles) into discrete screen fragments (potential pixels).

After clipping and culling, geometry is still made of mathematical lines and triangles. However, the screen is a grid of pixels. 

Rasterisation uses a **scanline algorithm** to determine which pixels are covered by each triangle:

- triangle vertices are sorted by $y$-coordinate to find the top, middle and bottom
- calculate the equation for the left and right edges
  - these are typically linear, in the form $Ax+By+C=0$
- for each horizontal row (scanline) in the pixel grid,
  - compute:
    - $x_{start}$: the x-coordinate where the scanline intersects the left edge
    - $x_{end}$: the x-coordinate where the scanline intersects the right edge
  - the pixels from $x_{start}$ to $x_{end}$ form a **span**
    - each pixel centre inside the span becomes a fragment
    - a fragment is a potential pixel, containing screen-space position $(x, y)$, depth ($z$-value) and attributes such as colour and texture
      - attributes are linearly interpolated between $x_{start}$ and $x_{end}$ to compute per-fragment values
  - generally, a pixel is only filled if its centre point lies inside the triangle

![Rasterisation](rasterisation.png)

Because pixels are discrete, triangle edges look jagged. This is called **aliasing**. It occurs because continuous shapes are approximated using a discrete pixel grid.

### Hidden surface removal (HSR) and shading

The final stage of the rendering pipeline determines:

- Which fragments are visible (Hidden Surface Removal)
- What colour they should be (Shading)

#### Shading

Shading is performed by the **fragment shader**. 

The fragment shader calculates the final colour of each fragment using:

- surface normals
- lighting (light sources, direction, intensity)
- textures (UV mapping)

For each fragment, it outputs a final pixel colour.

#### Hidden surface removal

Not all fragments should be drawn - some are hidden behind others.

##### Painter's algorithm

The historical approach was the Painter's algorithm:

- sort polygons by depth (approximate back-to-front ordering)
- draw far objects first, then paint nearer ones on top

However, this method had problems:

- it was slow, since sorting is $O(n \log n)$
- it fails completely for cyclic overlap, where three triangles overlap each other in a cycle (A in front of B, B in front of C, C in front of A)

##### Z-buffer

The modern solution is to use a **$Z$-buffer** (**depth buffer**):

- the GPU maintains the depth buffer, which is a 2D array of floats matching the screen resolution that stores the depth of the closest fragment seen so far
- at the start of every frame, the depth buffer is cleared to the maximum distance (e.g., $z = \infty$)
- then, for each frame, for each fragment at position $(x,y)$:
  - if $Z_{new} < Z_{stored}$:
    - update depth buffer with $Z_{new}$
    - write fragment to frame buffer (stores final pixel colour output of the image)
  - else, discard the fragment (as it is hidden)

![Painter's algorithm and Z-buffer](hsr.png)

This method is:

- fast - constant ($O(1)$) check time per pixel
- order independent - triangles can be processed in any order
- precise - visibility is decided per pixel (can handle cyclic overlaps)
- highly parallelisable - depth testing is performed independently per fragment, making it ideal for GPU architecture

However, a possible problem can occur if the depth buffer has limited precision:

- **Z-fighting** is when the GPU cannot reliably distinguish which fragment is closer, causing flickering

## GPU memory architecture

Beyond the processing pipeline, the GPU acts as a massive, high-speed memory manager. 

It manages distinct regions of VRAM (Video Random Access Memory) called **buffers**. Two of the most important things that it must manage are:

- **time** (to ensure the image is visually stable, with no flickering/tearing)
- **space** (to ensure the image is geometrically correct i.e., proper overlap of objects)

To manage space, the GPU uses $Z$-buffering, which was covered [above](#Z-buffer). 

### Managing time: double buffering

- the monitor refreshes line-by-line, from top to bottom
- if the GPU writes new pixels while the monitor is refreshing, the top of the screen may show the previous frame, while the bottom shows the new frame
- this is called **screen tearing**

The solution to this is to use **double buffering**, which decouples rendering from display, ensuring only complete frames are shown:

- separate GPU writing and display reading using two buffers:
  - **back buffer** (private): where the GPU renders the next frame (never visible)
  - **front buffer** (public): contains the completed frame read by the display
- when the GPU finishes drawing the back buffer
  - the buffers are swapped at the next **VSync**, or Vertical Sync (end of monitor refresh cycle)
    - back buffer becomes the new front buffer
    - GPU begins clearing and drawing to the old front buffer (new back buffer)
 
This prevents tearing, but may introduce input latency due to waiting for VSync.

### Memory cost

For each pixel, the buffers required are:

- front buffer (RGBA): 4 bytes
- back buffer (RGBA): 4 bytes
- depth buffer: 4 bytes

Therefore, the minimum VRAM required to render a frame is approximately given by:
$$ \text{VRAM} \approx \text{Pixels} \times 12 \ \text{bytes}$$

For 1080p (~2 million pixels), this is $\approx 24 \ \text{MB}$.

Note that this only the bare minimum cost to just open a window with the given number of pixels, before any textures, models etc. are loaded, which would significantly increase the required memory. 

## The modern programmable pipeline

Modern APIs like WebGL abstract the rendering pipeline into three main conceptual stages, centred on the programmable parts:

1. Vertex processing (programmable)
2. Rasterisation (fixed/ non-programmable)
3. Fragment processing (programmable)

![Rendering Pipeline](render_pipeline.png)

### Vertex processing

**Vertex processing** is the first programmable stage. Its primary purpose is to process the individual points (vertices) that make up the 3D models.

The GPU does this using the **vertex shader**:

- written by the programmer
- runs once for every vertex in the scene
- responsible for:
  - **transformation**: transforming vertex positions from model space through world and camera space into clip space (later converted to screen coordinates)
    -  involves a series of matrix multiplications that account for the model's position, the camera's viewpoint, and the perspective effect
  - **data passthrough**: passing per-vertex data (e.g. texture coordinates, normals) to the next stage

### Rasterisation

**Rasterisation** is a fixed, non-programmable part of the hardware. It takes transformed vertices from the previous stage and determines which pixels on the screen are covered by the geometric shapes (primitives) they form:

- vertices are assembled into primitives (usually triangles)
- rasteriser fills in these triangles, iterating over the 2D grid of pixels and generating a fragment for each pixel that lies inside the triangle's boundary
- per-vertex attributes (e.g. colour) are smoothly interpolated across the surface of the triangle for each fragment

### Fragment processing

**Fragment processing** is the second programmable stage.

The GPU performs this using the **fragment shader**:

- written by the programmer
- runs once for every fragment generated by the rasteriser
- computes the final colour of the fragment
- uses the interpolated data from the rasteriser to perform tasks like texturing and lighting

After this stage, per-fragment operations (e.g. depth testing) determine whether the fragment is written to the framebuffer (stores the colour and related data of pixels for a rendered frame before it is displayed).

## The polygon model

The **polygon model** represents 3D objects using meshes made of polygons, almost always triangles.
Triangles are the native primitive of the GPU.

There are several reasons for using triangles:

- **guaranteed flatness**:
  - a triangle is the simplest polygon, defined by exactly three points
  - any 3 points in 3D space are always coplanar (lie in the same plane)
  - therefore, a triangle is always perfectly flat
- **well-defined normals**:
  - a triangle has a single constant normal vector $\vec{N}$
  - this normal is perpendicular to the surface and constant across the triangle (since the triangle is flat)
- **precise lighting**:
  - lighting calculations depend on the surface normal
  - since triangles have a constant normal, lighting is simple and predictable

Another polygon, like a quad (4 vertices) is not guaranteed to be flat. If it is non-coplanar (bent), different parts of the surface face different directions, so it does not have a single consistent normal, leading to ambiguous lighting. In practice, quads are split into two triangles.

Modern GPUs are designed and optimised to process triangles; the rasterisation process assumes triangle input. 

### Triangle reduction

Rasterisation (converting shapes to pixels) and shading (calculating colour/light per pixel) are the most expensive stages of rendering. Therefore, the number of triangles reaching these stages should be minimised as early as possible in the pipeline. This is achieved using stages such as [clipping](#Clipping) and [backface culling](#Backface-culling).

## Transformations

A **transformation** is a mathematical operation that changes the position, rotation, or scale of geometric data. 

Transformations are needed as 3D models are just stored as static numbers, so to create a dynamic, interactive world, this data needs to be moved through different coordinate spaces (e.g. model $\to$ world $\to$ view $\to$ clip space). 

A naive approach may be to modify the vertex data directly on the CPU by iterating through each vertex and applying an operation. However, this is massively inefficient for many reasons:

- **High CPU Cost**: processing millions of vertices per frame is expensive, and the CPU must also handle tasks like game logic and physics
- **Slow Data Transfer**: sending updated vertex data to the GPU every frame is limited by bus bandwidth
- **Loss of Original Data**: modifying the data directly overwrites the original model, making reuse difficult

A much more efficient method is to perform transformations on the GPU in the vertex shader:

- the original vertex data is sent to the GPU once
- each frame, **transformation matrices** are sent to the GPU as uniform variables
  - these matrices encode transformations and are used to transform vertex positions
- the vertex shader applies these matrices to every vertex in parallel

This approach offloads massively parallel computation to the GPU, which is designed for it (having thousands of cores) and minimises the data transfer between the CPU and GPU. 

## Fundamental 2D transformations

The three fundamental 2D transformations that form the basis of most 2D graphics operations are:

- **translation**
- **rotation**
- **scaling**

### Translation (moving)

Translation is the process of moving an object from one position to another. It is defined by a translation vector $T = (t_x, t_y)$, which specifies the displacement in the $x$ and $y$ directions. 

For any vertex $P = (x, y)$, the translated vertex $P'$ is calculated by simple vector addition:
$$P' = P + T = (x + t_x, y + t_y)$$

![Translation](translation.png)

### Rotation (turning)

Rotation is the process of turning an object around a specific point, known as the pivot point. For simplicity, we first consider rotation around the origin $(0, 0)$. 

To rotate a vertex $P = (x, y)$ by an angle $\theta$ anti-clockwise, the following trigonometric formulae are used to find the new vertex $P'=(x',y')$:
$$
x' = x \cos \theta - y \sin \theta
$$
$$
y' = x \sin \theta + y \cos \theta
$$

![Rotation](rotation.png)

### Scaling (resizing)

Scaling is the process of changing the size of an object. It is defined by a scaling vector $S = (s_x, s_y)$. For a vertex $P = (x, y)$, the scaled vertex $P' = (x', y')$ is found by component-wise multiplication:
$$
x' = x \times s_x
$$
$$
y' = y \times y_x
$$

![Scaling](scaling.png)

If $s_x = s_y ~ $, the scaling is uniform, preserving the object's aspect ratio. If $s_x \neq s_y ~ $, the scaling is non-uniform, which will stretch or squash the object. Scaling is also performed relative to the origin; vertices move farther away from or closer to the origin based on the scaling factors.


### Transformations as matrices

For each of the three transformations above, the equations use different operations (addition for translation, trigonometric functions for rotation, multiplication for scaling).

Matrices provide a single, consistent mathematical operation to represent all transformations:
- each 2D point $(x,y)$ can be represented as a column vector
$
\begin{pmatrix}
x \\
y \\
\end{pmatrix}
$
- the transformation is represented as a matrix
- the transformation is applied by performing matrix-vector multiplication

#### Scaling matrix

The scaling operation $x' = x \times s_x, y' = y \times s_y$ can be written in matrix form as:
$$
\begin{pmatrix}
x' \\
y' \\
\end{pmatrix} =
\begin{pmatrix}
s_x & 0 \\
0 & s_y \\
\end{pmatrix}
\begin{pmatrix}
x \\
y \\
\end{pmatrix}
$$


#### Rotation matrix
The rotation operation $x' = x \cos \theta - y \sin \theta, y' = x \sin \theta + y \cos \theta$ can be written as:
$$
\begin{pmatrix}
x' \\
y' \\
\end{pmatrix} =
\begin{pmatrix}
\cos \theta & - \sin \theta \\
\sin \theta & \cos \theta \\
\end{pmatrix}
\begin{pmatrix}
x \\
y \\
\end{pmatrix}
$$

#### The problem with translation

Using the matrices above, scaling and rotation are now a single, consistent operation: multiplication by a $2 \times 2$ matrix. 

However, translation is an addition: $P' = P + T$. There is no $2 \times 2$ matrix $M$ such that:
$$
M
\begin{pmatrix}
x \\
y \\
\end{pmatrix} = 
\begin{pmatrix}
x + x_t\\
y + y_t\\
\end{pmatrix}
$$

This is because matrix multiplication is a linear transformation (origin stays fixed), but translation is an affine one (origin moves).

##### Homogeneous coordinates

To solve this problem, **homogeneous coordinates** can be used:

- the idea is to extend 2D coordinates into homogeneous coordinates using an extra dimension
- a 2D point $P = (x, y)$ can be represented as a 3D vector by adding a third '$w$' coordinate and setting it to 1:
$
\begin{pmatrix}
x \\
y \\
1 \\
\end{pmatrix}
$
- by moving to 3D vectors and $3 \times 3$ matrices, the third column of the matrix can be used to encode translation
- this allows affine transformations in 2D (like translation) to be represented as linear transformations in homogeneous coordinates
- $w$ acts as a scale factor, where the 'real world' is the slice where $w = 1$
- changing $w$ scales the coordinate values, but represents the same location in space (once normalised):
  - any point $(x, y, w)$ is equivalent to $(x/w, \ y/w, \ 1)$ - they lie on the same projective ray

Each transformation can now be represented as a $3 \times 3$ matrix which is multiplied with the homogeneous coordinate vector. 

#### Translation matrix (homogeneous)

$$
\begin{pmatrix}
x' \\
y' \\
1 \\
\end{pmatrix} =
\begin{pmatrix}
1 & 0 & t_x \\
0 & 1 & t_y \\
0 & 0 & 1 \\
\end{pmatrix}
\begin{pmatrix}
x \\
y \\
1 \\
\end{pmatrix}
$$

#### Rotation matrix (homogeneous)

$$
\begin{pmatrix}
x' \\
y' \\
1 \\
\end{pmatrix} =
\begin{pmatrix}
\cos \theta & - \sin \theta & 0\\
\sin \theta & \cos \theta & 0 \\
0 & 0 & 1
\end{pmatrix}
\begin{pmatrix}
x \\
y \\
1 \\
\end{pmatrix}
$$

#### Scaling matrix (homogeneous)

$$
\begin{pmatrix}
x' \\
y' \\
1 \\
\end{pmatrix} =
\begin{pmatrix}
s_x & 0 & 0 \\
0 & s_y & 0 \\
0 & 0 & 1 \\
\end{pmatrix}
\begin{pmatrix}
x \\
y \\
1 \\
\end{pmatrix}
$$

### Combining transformations

The key benefit of this unification is that it allows multiple transformations to be combined into a single matrix via matrix multiplication. 

It is very important to note that the order of matrix multiplication matters, as transformations are not commutative. In general, for matrices $A$ and $B$, $AB \neq BA$. 

Transformations are applied from right to left (when using column vectors): 

- for example, to first scale and object, then rotate it, and finally translate it, the matrices are applied in that order:
$$
P' = M_{translate} \times M_{rotate} \times M_{scale} \times P
$$

This is the conventional order of operations for placing on object (scale $\to$ rotate $\to$ translate).  This ensures the object scales in its own local space, rotates around its own centre, and then moves to its final world position.

The combined matrix can be computed once and applied to all vertices efficiently.

## 3D transformations

Moving from 2D to 3D is a natural extension of the above concepts. The principles remain the same, just with an extra dimension added:

- vertices now have three components: $(x,y,z)$
- homogeneous coordinates are used by adding a $w$ component, making the vectors 4D: $(x,y,z,1)$
- the transformation matrices are now $4 \times 4$

The process of taking a 3D model and rendering it onto a 2D screen is a journey through several distinct coordinate systems. This journey is orchestrated by a sequence of three critical matrices:

![3D to 2D Pipeline](3d_to_2d.png)

### The model matrix

The **model matrix** is responsible for taking a model's vertices in its local coordinate system (model space) and positioning them within the larger world. It applies a specific translation, rotation, and scale to the object to set its size, orientation, and position in the overall scene, transforming the vertices from model space into world space. Every object in a scene will have its own unique model matrix.

The model matrix is a combination of the 3D transformation matrices in homogeneous coordinates, which are direct extensions of their 2D counterparts. Rotation in 3D is more complex rotation can be around the X, Y, or Z axes, so there are three matrices for rotation. 

$$
M_{translate} =
\begin{pmatrix}
1 & 0 & 0 & t_x \\
0 & 1 & 0 & t_y \\
0 & 0 & 1 & t_z \\
0 & 0 & 0 & 1
\end{pmatrix}
$$

$$
M_{scale} =
\begin{pmatrix}
s_x & 0   & 0   & 0 \\
0   & s_y & 0   & 0 \\
0   & 0   & s_z & 0 \\
0   & 0   & 0   & 1
\end{pmatrix}
$$

$$
R_x(\theta) =
\begin{pmatrix}
1 & 0           & 0            & 0 \\
0 & \cos\theta  & -\sin\theta  & 0 \\
0 & \sin\theta  & \cos\theta   & 0 \\
0 & 0           & 0            & 1
\end{pmatrix}
$$

$$
R_y(\theta) =
\begin{pmatrix}
\cos\theta  & 0 & \sin\theta & 0 \\
0           & 1 & 0          & 0 \\
-\sin\theta & 0 & \cos\theta & 0 \\
0           & 0 & 0          & 1
\end{pmatrix}
$$

$$
R_z(\theta) =
\begin{pmatrix}
\cos\theta  & -\sin\theta & 0 & 0 \\
\sin\theta  & \cos\theta  & 0 & 0 \\
0           & 0           & 1 & 0 \\
0           & 0           & 0 & 1
\end{pmatrix}
$$